In [2]:
from tensorflow.keras.datasets import cifar10
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [3]:
(xtrain, ytrain), (xtest, ytest) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


In [4]:
xtrain = xtrain/255.0
xtest = xtest/255.0

In [11]:
class Cifar10Dataset(Dataset):
  def __init__(self, x, y):
    self.x = torch.tensor(x, dtype = torch.float32).permute(0, 3, 1, 2)
    self.y = torch.tensor(y, dtype=torch.long).squeeze()

  def __len__(self):
      return len(self.x)

  def __getitem__(self, idx):
      return self.x[idx], self.y[idx]

train_dataset = Cifar10Dataset(xtrain, ytrain)
test_dataset = Cifar10Dataset(xtest, ytest)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=128)

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
    self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.fc1 = nn.Linear(64 * 8 * 8, 256)
    self.fc2 = nn.Linear(256, 10)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = torch.flatten(x, 1)
    x = F.relu(self.fc1(x))
    return self.fc2(x)

In [15]:
import torch.optim as optim

model = CNN().cuda()

optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = optim.lr_scheduler.StepLR(
    optimizer, step_size=10, gamma=0.1
)

In [16]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [17]:
criterion = nn.CrossEntropyLoss()
early_stopping = EarlyStopping(patience=5)

for epoch in range(50):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Step the scheduler
    scheduler.step()

    # Early stopping check
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break

Epoch 1, Train Loss: 592.1176, Val Loss: 1.2225
Epoch 2, Train Loss: 443.6622, Val Loss: 1.0563
Epoch 3, Train Loss: 382.7247, Val Loss: 0.9601
Epoch 4, Train Loss: 339.0930, Val Loss: 0.9447
Epoch 5, Train Loss: 307.2122, Val Loss: 0.9157
Epoch 6, Train Loss: 277.9049, Val Loss: 0.8536
Epoch 7, Train Loss: 249.1942, Val Loss: 0.8746
Epoch 8, Train Loss: 223.2400, Val Loss: 0.8780
Epoch 9, Train Loss: 197.4849, Val Loss: 0.8999
Epoch 10, Train Loss: 172.7668, Val Loss: 0.9189
Epoch 11, Train Loss: 114.6066, Val Loss: 0.8975
Early stopping triggered!
